In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Break Signal Validation
NUM_BREAK_IDS = 500

np.random.seed(42)

X_train = pd.read_parquet('../data/X_train.parquet')
y_train_index = pd.read_parquet('../data/y_train_index.parquet')

# Random sample of break and no break series
break_ids = y_train_index[y_train_index['tau'] != -1].sample(n=NUM_BREAK_IDS).index
no_break_ids = y_train_index[y_train_index['tau'] == -1].sample(n=NUM_BREAK_IDS).index

# Get online segment
break_series = X_train.loc[break_ids].query("period == 2")
no_break_series = X_train.loc[no_break_ids].query("period == 2")

# How long each online segment is 

## Filter by length 
no_break_lengths = no_break_series.groupby(level='id').size()
no_break_lengths = no_break_lengths[no_break_lengths > 200]
no_break_ref_points = no_break_lengths.apply(lambda n: np.random.randint(100, n - 100))

break_lengths = break_series.groupby(level='id').size()
break_tau_index = y_train_index.loc[break_ids, 'tau_index']
valid_break_ids = break_tau_index[
    (break_tau_index >= 100) & ((break_lengths - break_tau_index) >= 100)
].index
break_lengths = break_lengths.loc[valid_break_ids]
break_tau_index = break_tau_index.loc[valid_break_ids]

print(len(break_tau_index))
print(len(no_break_ref_points))

248
400


In [ ]:
# Compute statistics for a series of values
def compute_stats(values):
    return pd.Series({
        'mean': values.mean(),
        'median': values.median(),
        'std': values.std(),
        'skew': values.skew(),
        'kurtosis': values.kurtosis(),
        'min': values.min(),
        'max': values.max(),
        'autocorrelation': values.autocorr(lag=1),
    })


def compute_shift_magnitude(serie, ref_point):
    before = serie.iloc[ref_point - 100:ref_point]['value']
    after = serie.iloc[ref_point:ref_point + 100]['value']
    return (compute_stats(after) - compute_stats(before)).abs()


break_diffs = pd.DataFrame({
    i: compute_shift_magnitude(break_series.loc[i], break_tau_index[i])
    for i in break_tau_index.index # every id 
}).T


no_break_diffs = pd.DataFrame({
    i: compute_shift_magnitude(no_break_series.loc[i], no_break_ref_points[i])
    for i in no_break_ref_points.index
}).T

summary = pd.DataFrame({
    'Break (avg |Δ|)': break_diffs.mean(),
    'No Break (avg |Δ|)': no_break_diffs.mean(),
})
summary['Difference'] = summary['Break (avg |Δ|)'] - summary['No Break (avg |Δ|)']
display(summary)


,Break (avg |Δ|),No Break (avg |Δ|),Difference
mean,0.136251,0.130222,0.006029
median,0.156552,0.151827,0.004725
std,0.206194,0.132591,0.073603
skew,0.548852,0.487910,0.060942
kurtosis,1.976092,1.879735,0.096358
min,0.987796,0.769830,0.217966
max,0.943753,0.791759,0.151994
